# IM Grade 4 — organize the official ZIP archives in Drive\nRun the cell below in a Google Colab notebook. It will request authorization to mount your Google Drive, then create one folder per IM unit and preserve the original ZIPs.\nThe large lesson archive may take a while; do not close the tab during extraction.\n

In [ ]:
# Run in Google Colab. Mount Drive, then safely unpack the official Grade 4 IM ZIPs.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile, shutil, re, os, time

DRIVE = Path('/content/drive/MyDrive')
ARCHIVES = {
 'Grade4-downloadable_materials-pdf.zip': 'Lesson Materials',
 'Grade4-Course-teacher_guide-Unit-pdf.zip': 'Teacher Guides',
 'Grade4-Course-family_support_materials-Unit-pdf.zip': 'Family Support',
 'Grade4-Course-center_overview-Unit-pdf.zip': 'Centers',
 'Grade4-Course-unit_at_a_glance-Unit-pdf.zip': 'Unit At A Glance',
 'Grade4-Course-monitoring_sheet-Unit-pdf.zip': 'Monitoring Sheets',
}
# Set the folder explicitly if your files are not found through MyDrive search:
SOURCE = None  # e.g. Path('/content/drive/MyDrive/Curriculum Library/4th Grade/Math')
DESTINATION = None # defaults to SOURCE / 'IM Grade 4 - Extracted'

def discover_source():
    if SOURCE is not None:
        return Path(SOURCE)
    matches = []
    for directory, subdirs, filenames in os.walk(DRIVE):
        depth=len(Path(directory).relative_to(DRIVE).parts)
        if depth >= 8:
            subdirs[:] = []
        if sum(name in filenames for name in ARCHIVES)>=3:
            matches.append(Path(directory))
    if len(matches)!=1:
        raise RuntimeError(f'Expected one 4th-grade Math archive folder; found {matches}. Set SOURCE explicitly.')
    return matches[0]

source=discover_source()
dest=Path(DESTINATION) if DESTINATION else source/'IM Grade 4 - Extracted'
dest.mkdir(parents=True,exist_ok=True)
print('Source:',source)
print('Destination:',dest)
summary=[]
for archive_name,category in ARCHIVES.items():
    original=source/archive_name
    if not original.exists():
        print('NOT FOUND:',archive_name)
        summary.append((archive_name,0,'missing'))
        continue
    local=Path('/content')/archive_name
    print('\nCopying:',archive_name)
    shutil.copyfile(original,local) # local SSD avoids slow Drive-mounted random ZIP reads
    done=0
    try:
        with zipfile.ZipFile(local) as z:
            entries=[a for a in z.infolist() if not a.is_dir()]
            total=sum(a.file_size for a in entries)
            if len(entries)>30000 or total>20_000_000_000:
                raise RuntimeError('Unusually large archive expansion: review ZIP before extracting.')
            print('Entries:',len(entries),'Uncompressed bytes:',total)
            for info in entries:
                # Never trust paths stored inside an archive.
                pure=Path(info.filename.replace('\\','/'))
                basename=pure.name
                if not basename or not basename.lower().endswith('.pdf'):
                    continue
                m=re.search(r'Grade4[-_ ](\d+)',basename,re.I)
                if not m:
                    print('Skipped unclassified entry:',basename)
                    continue
                unit=int(m.group(1))
                if not 1<=unit<=9:continue
                target=dest/f'Unit {unit:02d}'/category
                sec=re.search(r'Section[-_ ]([A-D])\b',basename,re.I)
                lesson=re.search(r'Lesson[-_ ](\d+)\b',basename,re.I)
                if sec:
                    target=target/f'Section {sec.group(1).upper()}'
                if lesson:
                    target=target/f'Lesson {int(lesson.group(1)):02d}'
                target.mkdir(parents=True,exist_ok=True)
                final=target/basename
                if final.exists() and final.stat().st_size==info.file_size:
                    continue
                if final.exists():
                    final=target/(Path(basename).stem+'_another.pdf')
                # Streaming copy keeps RAM use low even for large PDFs.
                with z.open(info) as src, open(final,'wb') as out:
                    shutil.copyfileobj(src,out,1024*1024)
                done+=1
                if done%50==0:print('Extracted',done,'from',archive_name)
        summary.append((archive_name,done,'ok'))
    finally:
        try:local.unlink()
        except Exception:pass
print('\nCompleted. Files newly extracted:')
for name,count,status in summary:
    print(f'  {status}: {count:4d} — {name}')
print('Original ZIP files were NOT deleted or changed.')
